In [ ]:
from src.slm_initialise import slm_initialise
from src.utils import *
from src.plots import plot_fit,bb,g
from slmsuite.holography.algorithms import FeedbackHologram
from slmsuite.holography.toolbox import convert_vector
from slmsuite.holography.toolbox.phase import zernike_sum,zernike_get_string,ZERNIKE_NAMES 
from src.algos import *
from src.config import base_path
from src.zernike import get_zernike_calibrations,get_cholo_phase,get_cholo_bb_g_phase,fit_metric,fit_metric_udf,plot_parabola,get_zernike_dict_from_metrics_df,get_zernike_metrics_df,zernike_names,get_tweezer_metrics
from src import global_store
import pickle
import pandas as pd
%load_ext autoreload
%autoreload 2

In [ ]:
fs = slm_initialise(force=True,verbose=True)
calibrate(fs,vendor = False,refresh = False)
sp_calibrate(fs,refresh=False)
plot_cam(fs,phase="zero",centre=True, save=True)

# Zernike testing

Iterate a single zernike polynomial and get the cam images
(shift the zernike polynomials to the slm centre)
visualise the plots to see the effect of aberration
calculate metric and visualise for convexity or atleast a minima

for a given zernike term
    for a given spot 
        find fit metrics (rmse error??)

## Code to iterate zernikes and get the zernike calibrations

In [ ]:
target_shape = fs.cam.shape
j,i = np.meshgrid(np.arange(target_shape[0]), np.arange(target_shape[1]), indexing='ij')

centrex = 1180
centrey = 230
n_spots = 5
r = 50
n = 4
rad = 100
bb_locations = np.array([[centrex+(i-2)*rad ,centrey + (j-1)*rad] for i in range(n) for j in range(n)]).T
spot_locations = np.array([[centrex+(i-2)*rad + 50,centrey + (j-1)*rad + 50] for i in range(n) for j in range(n)])

zernike_calibrations = []
nz = 20
for i in range(len(spot_locations)):
    zernike_xy = np.flip(toolbox.convert_vector([1440+10,1024+65] - spot_locations[i],from_units="ij",to_units="zernike",hardware=fs).T[0])
    zernike_arr = np.concatenate([zernike_xy, np.zeros(nz -2, dtype=zernike_xy.dtype)])
    zernike_calibrations.append({i+1: float(v) for i, v in enumerate(zernike_arr)})
zernike_indices = [i+1 for i in range(nz)]

In [ ]:
phases = []
calibs = []
calibs_full = []
for i in range(5):
    for zi in range(1,nz+1):
        file_name = f"zernike_pert_test_iter_{i}_{zi}"
        phase = get_cholo_phase(zernike_calibrations=zernike_calibrations,fs = fs)
        zernike_calibrations_new = get_zernike_calibrations(phase,spot_locations,[zi],file_name)
        zernike_calibrations = [{k: d1[k] -  d2.get(k, 0) for k in d1} for d1, d2 in zip(zernike_calibrations, zernike_calibrations_new)]
        phases.append(phase)
        calibs.append(zernike_calibrations_new)
        calibs_full.append(zernike_calibrations)


# Plotting rmse/streshlprime vs zernike perturbation weight for each zernike


In [ ]:
import pandas as pd
s=0
zi = 4
file_name = f"zernike_pert_test_iter_{0}_{zi}"
load_path = base_path / "data" / "zernike" / "raw" / f"{file_name}_df.csv"
pdf = pd.read_csv(load_path)

subset = pdf[
    (pdf['zernike_index'] == zi) &
    (pdf['expected_y'] == spot_locations[s][1]) &
    (pdf['expected_x'] == spot_locations[s][0])
]

ax = subset.plot(x="w", y="r_squared", label="r_squared", color="blue")
subset.plot(x="w", y="strehl_prime", label="strehl_prime", color="red", ax=ax, secondary_y=True)